# StoryForge - Personalized Children's Book Studio

```text
📚 STORYFORGE - Personalized Children's Book Studio
================================================================
Input : 3 photos of your child + a theme + a name
Output: ./storyforge_out/
        ├── page_1.png ... page_N.png  (child = hero, consistent face)
        ├── page_1_narration.wav ...   (warm bedtime narration)
        ├── cover_animated.mp4         (animated book cover)
        └── storybook.pdf              (print-ready, assembled by
                                        Gemini's code-execution sandbox)

Models:
  gemini-3.1-pro-preview        -> structured storybook screenplay + PDF code
  gemini-3-pro-image-preview    -> illustrations w/ multi-reference consistency
  gemini-3.1-flash-tts-preview  -> narration audio
  gemini-omni-1.1-flash         -> animated cover video

Install: pip install -U google-genai pydantic pillow
Auth   : export GEMINI_API_KEY="..."
```

In [1]:
import os
import time
import wave
from pydantic import BaseModel, Field
from PIL import Image as PILImage
from google import genai
from google.genai import types
from google.colab import userdata

In [2]:
GEMINI_API_KEY = userdata.get("GEMINI_KEY")

In [3]:
client = genai.Client(api_key=GEMINI_API_KEY)

In [5]:
for m in client.models.list():
    print(f"Model: {m.name}")

Model: models/gemini-2.5-flash
Model: models/gemini-2.5-pro
Model: models/gemini-2.5-flash-preview-tts
Model: models/gemini-2.5-pro-preview-tts
Model: models/gemma-4-26b-a4b-it
Model: models/gemma-4-31b-it
Model: models/gemini-flash-latest
Model: models/gemini-flash-lite-latest
Model: models/gemini-pro-latest
Model: models/gemini-2.5-flash-lite
Model: models/gemini-2.5-flash-image
Model: models/gemini-3-flash-preview
Model: models/gemini-3.1-pro-preview
Model: models/gemini-3.1-pro-preview-customtools
Model: models/gemini-3.1-flash-lite-preview
Model: models/gemini-3.1-flash-lite
Model: models/gemini-3-pro-image-preview
Model: models/gemini-3-pro-image
Model: models/nano-banana-pro-preview
Model: models/gemini-3.1-flash-image-preview
Model: models/gemini-3.1-flash-image
Model: models/gemini-3.1-flash-lite-image
Model: models/gemini-3.5-flash
Model: models/gemini-3.5-flash-lite
Model: models/gemini-omni-flash-preview
Model: models/gemini-omni-1.1-flash
Model: models/gemini-3.5-transcrib

In [6]:
OUT = "storyforge_out"
os.makedirs(OUT, exist_ok=True)

In [7]:
WRITER  = "gemini-3.1-pro-preview"
ARTIST  = "gemini-3-pro-image-preview"
TTS     = "gemini-3.1-flash-tts-preview"
ANIMATOR = "veo-3.1-generate-preview"

In [8]:
# -----------------------------------------------------------------
# STEP 1 - Structured storybook from Gemini Pro
# -----------------------------------------------------------------

class Page(BaseModel):
    page_number: int
    story_text: str = Field(description="2-3 sentences, read-aloud rhythm, "
                                        "age 4-7 vocabulary")
    illustration_prompt: str = Field(
        description="Full scene description: setting, hero's action, "
                    "emotion, lighting. Refer to hero as 'the child from "
                    "the reference photos'.")
    narration_direction: str = Field(
        description="Voice direction, e.g. 'whisper with wonder', "
                    "'giggly and fast'")

class Storybook(BaseModel):
    title: str
    dedication: str
    art_style: str = Field(description="One locked style for every page")
    cover_prompt: str
    cover_motion: str = Field(description="How the cover should animate")
    pages: list[Page]

def write_storybook(child_name: str, theme: str,
                    photo_paths: list[str]) -> Storybook:
    photos = [PILImage.open(p) for p in photo_paths]
    resp = client.models.generate_content(
        model=WRITER,
        contents=[
            *photos,
            f"""Look at these photos of {child_name}. Write a magical
            6-page children's storybook where {child_name} is the hero.
            Theme: {theme}.
            Weave in real visual details you can see in the photos (hair,
            smile, favorite colors of clothing) so the story feels truly
            personal. Pick ONE art style (e.g. 'soft watercolor storybook')
            and lock it into art_style and every illustration_prompt."""
        ],
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="high"),
            response_mime_type="application/json",
            response_schema=Storybook,
        ),
    )
    return resp.parsed

In [9]:
# -----------------------------------------------------------------
# STEP 2 - Illustrations with locked character consistency
# Multi-turn CHAT with the image model: the SDK preserves the
# thought signatures between turns, which is what keeps the hero's
# face consistent from page to page.
# -----------------------------------------------------------------

def illustrate_book(book: Storybook, photo_paths: list[str],
                    child_name: str) -> list[str]:
    ref_photos = [PILImage.open(p) for p in photo_paths]  # ≤5 human refs

    chat = client.chats.create(
        model=ARTIST,
        config=types.GenerateContentConfig(
            response_modalities=["TEXT", "IMAGE"],
            image_config=types.ImageConfig(aspect_ratio="4:3"),
        ),
    )

    # Turn 0: lock the character design from the reference photos
    chat.send_message([
        *ref_photos,
        f"These are reference photos of {child_name}, the hero of a "
        f"children's book. Create a character design sheet of "
        f"{child_name} in this art style: {book.art_style}. Keep the "
        f"face, hair and likeness clearly recognizable from the photos.",
    ])

    paths = []
    for page in book.pages:
        resp = chat.send_message(
            f"Perfect. Now illustrate page {page.page_number}, keeping "
            f"{child_name} EXACTLY consistent with the character sheet: "
            f"{page.illustration_prompt}. Style: {book.art_style}. "
            f"No text in the image.")
        path = f"{OUT}/page_{page.page_number}.png"
        for part in resp.parts:
            if part.inline_data:
                part.as_image().save(path)
        paths.append(path)
        print(f"🎨 page {page.page_number} -> {path}")

    # Cover (with title text - Pro Image excels at legible text)
    resp = chat.send_message(
        f"Finally, the book COVER: {book.cover_prompt}. Render the title "
        f"'{book.title}' in beautiful hand-lettered storybook typography. "
        f"Same art style, same {child_name}.")
    cover = f"{OUT}/cover.png"
    for part in resp.parts:
        if part.inline_data:
            part.as_image().save(cover)
    print(f"🎨 cover -> {cover}")
    return [cover] + paths

In [10]:
# -----------------------------------------------------------------
# STEP 3 - Narration audio per page
# -----------------------------------------------------------------

def narrate_book(book: Storybook, voice: str = "Aoede"):
    for page in book.pages:
        resp = client.models.generate_content(
            model=TTS,
            contents=(f"Read this children's book page aloud. "
                      f"Direction: {page.narration_direction}. "
                      f"Text: {page.story_text}"),
            config=types.GenerateContentConfig(
                response_modalities=["AUDIO"],
                speech_config=types.SpeechConfig(
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(
                            voice_name=voice))),
            ),
        )
        pcm = resp.candidates[0].content.parts[0].inline_data.data
        path = f"{OUT}/page_{page.page_number}_narration.wav"
        with wave.open(path, "wb") as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(24000)
            wf.writeframes(pcm)
        print(f"🎙️ narration {page.page_number} -> {path}")

In [11]:
# -----------------------------------------------------------------
# STEP 4 - Animated cover video (image -> video)
# -----------------------------------------------------------------

def animate_cover(book: Storybook, cover_path: str):
    op = client.models.generate_videos(
        model=ANIMATOR,
        prompt=(f"Gentle magical children's book cover animation: "
                f"{book.cover_motion}. Soft sparkles, subtle parallax, "
                f"keep the hero's face unchanged."),
        image=types.Image.from_file(location=cover_path),
        config=types.GenerateVideosConfig(resolution="720p"),
    )
    while not op.done:
        time.sleep(10)
        op = client.operations.get(op)
    vid = op.response.generated_videos[0]
    path = f"{OUT}/cover_animated.mp4"
    client.files.download(file=vid.video)
    vid.video.save(path)
    print(f"🎥 animated cover -> {path}")

In [12]:
# -----------------------------------------------------------------
# STEP 5 - Print-ready PDF assembled by Gemini's CODE EXECUTION
# We pass the page images into the sandbox; Pro writes & runs the
# layout code and returns the finished PDF as inline bytes.
# -----------------------------------------------------------------

def assemble_pdf(book: Storybook, image_paths: list[str]):
    parts = [types.Part.from_bytes(
                data=open(p, "rb").read(), mime_type="image/png")
             for p in image_paths]
    texts = "\n".join(f"PAGE {p.page_number}: {p.story_text}"
                      for p in book.pages)

    resp = client.models.generate_content(
        model=WRITER,
        contents=[
            *parts,
            f"""Using code execution, write and RUN Python that assembles
            these images into a print-ready A4-landscape PDF storybook:
            - Image 1 is the cover (full bleed).
            - Each following image is a page; place the matching text
              below it in a large friendly font with generous margins.
            - Last page: the dedication '{book.dedication}'.
            Page texts:
            {texts}
            Return the finished PDF file."""
        ],
        config=types.GenerateContentConfig(
            tools=[types.Tool(code_execution=types.ToolCodeExecution())],
        ),
    )
    saved = False
    for part in resp.candidates[0].content.parts:
        if part.inline_data and "pdf" in (part.inline_data.mime_type or ""):
            with open(f"{OUT}/storybook.pdf", "wb") as f:
                f.write(part.inline_data.data)
            saved = True
            print(f"📄 PDF -> {OUT}/storybook.pdf")

    if not saved:   # graceful local fallback - Pillow can write PDFs
        imgs = [PILImage.open(p).convert("RGB") for p in image_paths]
        imgs[0].save(f"{OUT}/storybook.pdf", save_all=True,
                     append_images=imgs[1:])
        print(f"📄 PDF (fallback assembly) -> {OUT}/storybook.pdf")

In [13]:
# -----------------------------------------------------------------
# PIPELINE
# -----------------------------------------------------------------

def forge_story(child_name: str, theme: str, photo_paths: list[str]):
    print(f"\n📚 STORYFORGE - starring {child_name}!\n" + "-" * 55)
    book = write_storybook(child_name, theme, photo_paths)
    print(f"\n✨ '{book.title}' - style: {book.art_style}\n")
    images = illustrate_book(book, photo_paths, child_name)
    narrate_book(book)
    animate_cover(book, images[0])
    assemble_pdf(book, images)
    print(f"\n✅ Storybook complete -> ./{OUT}/")

In [14]:
if __name__ == "__main__":
    forge_story(
        child_name="Dhairya",
        theme="a brave astronaut who befriends a shy comet",
        photo_paths=["image_1.jpg", "image_2.jpg", "image_3.jpg"],
    )


📚 STORYFORGE - starring Dhairya!
-------------------------------------------------------

✨ 'Dhairya and the Shy Little Comet' - style: Soft watercolor storybook style with glowing, magical celestial accents

🎨 page 1 -> storyforge_out/page_1.png
🎨 page 2 -> storyforge_out/page_2.png
🎨 page 3 -> storyforge_out/page_3.png
🎨 page 4 -> storyforge_out/page_4.png
🎨 page 5 -> storyforge_out/page_5.png
🎨 page 6 -> storyforge_out/page_6.png
🎨 cover -> storyforge_out/cover.png
🎙️ narration 1 -> storyforge_out/page_1_narration.wav
🎙️ narration 2 -> storyforge_out/page_2_narration.wav
🎙️ narration 3 -> storyforge_out/page_3_narration.wav
🎙️ narration 4 -> storyforge_out/page_4_narration.wav
🎙️ narration 5 -> storyforge_out/page_5_narration.wav
🎙️ narration 6 -> storyforge_out/page_6_narration.wav
🎥 animated cover -> storyforge_out/cover_animated.mp4
📄 PDF -> storyforge_out/storybook.pdf

✅ Storybook complete -> ./storyforge_out/
